C1 — clone + patches + CORRECTED per-path dump

In [1]:
import subprocess, re, pathlib
subprocess.run("mkdir -p /kaggle/temp && cd /kaggle/temp && rm -rf FreeFine && "
               "git clone -q https://github.com/CIawevy/FreeFine.git", shell=True, check=True)
root = pathlib.Path("/kaggle/temp/FreeFine/evaluation/metrics")
mp = root/"main.py"; mp.write_text(mp.read_text().replace("args.3d","getattr(args, '3d')"))
for f in [root/"MD"/"mean_distance.py", root/"MD"/"dift_sd.py"]:
    f.write_text(f.read_text().replace("stabilityai/stable-diffusion-2-1","sd2-community/stable-diffusion-2-1"))

md = root/"MD"/"mean_distance.py"; src = md.read_text()
m = re.search(r'^([ \t]*)all_dist = \[\]', src, re.M); assert m, "all_dist anchor missing"; ind=m.group(1)
src = src.replace(m.group(0), f"{m.group(0)}\n{ind}_PER_KP = []", 1)
anchor = "s_img_path, t_img, s_mask, edit_param, prompt = data_pair"
mu = re.search(r'^([ \t]*)'+re.escape(anchor), src, re.M); assert mu, "unpack anchor missing"; iu=mu.group(1)
src = src.replace(mu.group(0), f"{mu.group(0)}\n{iu}_gen_path = t_img", 1)          # capture PATH before reassignment
assert src.count("all_dist.append(dist)") >= 1, "append anchor missing"
src = src.replace("all_dist.append(dist)", "all_dist.append(dist); _PER_KP.append((_gen_path, float(dist)))", 1)
m2 = re.search(r'^([ \t]*)md = torch\.tensor\(all_dist\)\.mean\(\)\.item\(\)', src, re.M); assert m2, "md= anchor missing"; k=m2.group(1)
dump=(f"{m2.group(0)}\n{k}import os as _os, csv as _csv\n{k}_dp=_os.environ.get('MD_DUMP_PATH')\n"
      f"{k}if _dp:\n{k}    with open(_dp,'w',newline='') as _f:\n"
      f"{k}        _w=_csv.writer(_f); _w.writerow(['gen','dist']); _w.writerows(_PER_KP)\n"
      f"{k}    print(f'MD per-kp dumped: {{len(_PER_KP)}} rows', flush=True)")
src = src.replace(m2.group(0), dump, 1)
md.write_text(src)
print("patched. verify the dump records the PATH variable:")
for ln in src.splitlines():
    if "_gen_path" in ln or "_PER_KP.append" in ln: print("   >>", ln.strip())

patched. verify the dump records the PATH variable:
   >> _gen_path = t_img
   >> all_dist.append(dist); _PER_KP.append((_gen_path, float(dist)))


C2 — metric_env (with the datasets<3 fix)


In [2]:
%%bash
set -e
pip install -q --root-user-action=ignore uv
uv python install 3.10.13
VENV=/kaggle/temp/metric_env; PY=$VENV/bin/python; REPO=/kaggle/temp/FreeFine
rm -rf $VENV && uv venv --python 3.10.13 $VENV
uv pip install --python $PY torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
uv pip install --python $PY "setuptools<70" wheel pip
grep -vi '^clip' $REPO/evaluation/metrics/requirements.txt > /tmp/metric_req.txt
uv pip install --python $PY -r /tmp/metric_req.txt
uv pip install --python $PY "setuptools<70"
uv pip install --python $PY --no-build-isolation "clip @ git+https://github.com/openai/CLIP.git@dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1"
uv pip install --python $PY "pyarrow<16" "datasets<3"
wget -q https://dl.fbaipublicfiles.com/mmf/clip/bpe_simple_vocab_16e6.txt.gz -P /tmp
for d in $(find $VENV -path '*/site-packages/clip' -o -path '*open_clip' -type d); do cp /tmp/bpe_simple_vocab_16e6.txt.gz "$d/" 2>/dev/null || true; done
echo "metric_env ready -> datasets $($PY -c 'import datasets; print(datasets.__version__)')"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 72.9 MB/s eta 0:00:00
metric_env ready -> datasets 2.21.0


 Downloaded cpython-3.10.13-linux-x86_64-gnu (download)
Installed Python 3.10.13 in 1.31s
 + cpython-3.10.13-linux-x86_64-gnu (python3.10)
Using CPython 3.10.13
Creating virtual environment at: /kaggle/temp/metric_env
Activate with: source /kaggle/temp/metric_env/bin/activate
Using Python 3.10.13 environment at: /kaggle/temp/metric_env
Resolved 27 packages in 567ms
 Downloaded nvidia-cuda-cupti-cu12
 Downloaded torchaudio
 Downloaded torchvision
 Downloaded nvidia-nvjitlink-cu12
 Downloaded nvidia-cuda-nvrtc-cu12
 Downloaded pillow
 Downloaded nvidia-curand-cu12
 Downloaded networkx
 Downloaded triton
 Downloaded numpy
 Downloaded nvidia-cusolver-cu12
 Downloaded nvidia-cusparselt-cu12
 Downloaded nvidia-nccl-cu12
 Downloaded nvidia-cufft-cu12
 Downloaded nvidia-cusparse-cu12
 Downloaded sympy
 Downloaded nvidia-cublas-cu12
 Downloaded nvidia-cudnn-cu12
 Downloaded torch
Prepared 27 packages in 44.33s
Installed 27 packages in 382ms
 + filelock==3.29.0
 + fsspec==2026.4.0
 + jinja2==3.1

C3 — link GeoBench from the cache (no HF download)


In [3]:
import os, glob, shutil
GEO="/kaggle/temp/GeoBenchMeta"; os.makedirs(f"{GEO}/Geo-Bench-2D", exist_ok=True)
shutil.copy(glob.glob("/kaggle/input/**/annotation_2d.json", recursive=True)[0], f"{GEO}/annotation_2d.json")
for sub in glob.glob("/kaggle/input/**/Geo-Bench-2D/*", recursive=True):
    if not os.path.isdir(sub): continue
    name=os.path.basename(sub); dst=f"{GEO}/Geo-Bench-2D/{name}"
    if os.path.islink(dst): os.remove(dst)
    elif os.path.isdir(dst): shutil.rmtree(dst)
    os.symlink(sub, dst)
print("linked:", sorted(os.listdir(f"{GEO}/Geo-Bench-2D")))   # expect source_img, source_mask, ...

linked: ['source_img', 'source_img_full_v2', 'source_mask', 'target_mask']


C4 — symlink gen (FINAL folder) + reconstruct full manifest


In [4]:
import os, glob, json, shutil
GEO="/kaggle/temp/GeoBenchMeta"
PNG_ROOT = glob.glob("/kaggle/input/**/gen_results_2d_final/gen_results_2d_backup", recursive=True)[0]
print("gen root:", PNG_ROOT)                                   # MUST say gen_results_2d_final
link=f"{GEO}/Gen_results_FreeFine_2d"
if os.path.islink(link): os.remove(link)
elif os.path.isdir(link): shutil.rmtree(link)
os.symlink(PNG_ROOT, link)
print("gen images:", len(glob.glob(f"{link}/**/*.png", recursive=True)))   # expect 5677
man=f"{GEO}/generated_results_freefine_2d.json"
ann=json.load(open(f"{GEO}/annotation_2d.json")); added=0
for d,da in ann.items():
    for i,ins in da.get("instances",{}).items():
        for e in list(ins):
            rel=f"Gen_results_FreeFine_2d/{d}/{i}/{e}.png"
            if os.path.exists(os.path.join(GEO,rel)):
                ins[e]["gen_img_path"]=rel; added+=1
json.dump(ann,open(man,"w")); print("manifest entries:", added)   # expect 5677

gen root: /kaggle/input/datasets/georgiostzamouranis/freefine-geobench2d-bggen/gen_results_2d_final/gen_results_2d_backup
gen images: 5677
manifest entries: 5677


C5 — 2-GPU MD shard + dump + combine


In [5]:
import os, re, json, time, select, subprocess, csv
GEO="/kaggle/temp/GeoBenchMeta"; MET="/kaggle/temp/FreeFine/evaluation/metrics"
PY="/kaggle/temp/metric_env/bin/python"; os.environ.setdefault("HF_HOME","/kaggle/temp/hf")
from huggingface_hub import snapshot_download
for a in range(6):
    try:
        snapshot_download("sd2-community/stable-diffusion-2-1", token=os.environ.get("HF_TOKEN"),
            allow_patterns=["*.json","*.txt","tokenizer/*","scheduler/*","feature_extractor/*",
                            "text_encoder/*.bin","unet/*.bin","vae/*.bin"]); print("SD-2.1 cached",flush=True); break
    except Exception as e: print("retry SD-2.1:",str(e)[:80],flush=True); time.sleep(30)
data=json.load(open(f"{GEO}/generated_results_freefine_2d.json"))
leaves=[(d,i,c) for d,da in data.items() for i,ins in da["instances"].items() for c in ins]
mid=len(leaves)//2; subs=[set(leaves[:mid]), set(leaves[mid:])]
print(f"{len(leaves)} cases -> {len(subs[0])}/{len(subs[1])}",flush=True)
def build(s):
    o={}
    for d,da in data.items():
        ni={i:{c:v for c,v in ins.items() if (d,i,c) in s} for i,ins in da["instances"].items()}
        ni={i:k for i,k in ni.items() if k}
        if ni: nd={k:v for k,v in da.items() if k!="instances"}; nd["instances"]=ni; o[d]=nd
    return o
half=[f"/kaggle/temp/md_half_{i}.json" for i in (0,1)]; dump=[f"/kaggle/working/md_kp_{i}.csv" for i in (0,1)]
for i,s in enumerate(subs): json.dump(build(s),open(half[i],"w"))
def cmd(p): return [PY,"main.py","--path",p,"--use_relative_path","--base_dir",GEO,
    "--fid_path",f"{GEO}/Geo-Bench-2D/source_img_full_v2","--task","000000100","--level","0"]
base=os.environ.copy(); base.update({"PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True","MPLBACKEND":"Agg","PYTHONUNBUFFERED":"1","HF_HOME":"/kaggle/temp/hf"})
procs,bufs=[],["",""]
for g in (0,1):
    e=base.copy(); e["CUDA_VISIBLE_DEVICES"]=str(g); e["MD_DUMP_PATH"]=dump[g]
    procs.append(subprocess.Popen(cmd(half[g]),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,env=e,cwd=MET,bufsize=0))
print("MD launched\n",flush=True)
fds={p.stdout.fileno():i for i,p in enumerate(procs)}; openf=set(fds); start=last=time.time()
def prog(b):
    m=re.findall(r"(\d+)/(\d+)\s*\[([^\]]*)\]",b[-4000:])
    if m: c,t,info=m[-1]; return f"{c}/{t} [{info}]"
    ls=[l for l in b[-2000:].replace("\r","\n").splitlines() if l.strip()]; return ls[-1][:70] if ls else "(loading...)"
while openf:
    for fd in select.select(list(openf),[],[],2.0)[0]:
        ch=os.read(fd,65536)
        if not ch: openf.discard(fd); continue
        bufs[fds[fd]]+=ch.decode("utf-8","replace")
    if time.time()-last>=30 or not openf:
        el=int(time.time()-start); print(f"[t={el//60}m{el%60:02d}s] G0:{prog(bufs[0])} || G1:{prog(bufs[1])}",flush=True); last=time.time()
for p in procs: p.wait()
rows=[]
for d in dump:
    if os.path.exists(d):
        with open(d) as f: r=csv.reader(f); next(r,None); rows+=[(g,float(x)) for g,x in r]
with open("/kaggle/working/md_per_kp_full.csv","w",newline="") as f:
    w=csv.writer(f); w.writerow(["gen","dist"]); w.writerows(rows)
print(f"\nper-kp rows:{len(rows)} | POOLED MD (exact) = {sum(x for _,x in rows)/len(rows):.6f}  [salvage said 8.4993]",flush=True)

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

SD-2.1 cached
5677 cases -> 2838/2839
MD launched

[t=0m30s] G0:-----MD----- || G1:-----MD-----
[t=1m01s] G0:2/2838 [00:16<6:24:01,  8.12s/it] || G1:2/2839 [00:16<6:21:08,  8.06s/it]
[t=1m32s] G0:6/2838 [00:44<5:42:07,  7.25s/it] || G1:6/2839 [00:45<5:46:46,  7.34s/it]
[t=2m03s] G0:10/2838 [01:13<5:44:07,  7.30s/it] || G1:10/2839 [01:15<6:00:40,  7.65s/it]
[t=2m34s] G0:14/2838 [01:46<6:11:22,  7.89s/it] || G1:14/2839 [01:49<6:28:38,  8.25s/it]
[t=3m05s] G0:18/2838 [02:18<6:16:06,  8.00s/it] || G1:17/2839 [02:15<6:43:57,  8.59s/it]
[t=3m36s] G0:22/2838 [02:49<6:06:33,  7.81s/it] || G1:21/2839 [02:48<6:31:29,  8.34s/it]
[t=4m07s] G0:26/2838 [03:19<5:53:21,  7.54s/it] || G1:25/2839 [03:22<6:32:36,  8.37s/it]
[t=4m39s] G0:30/2838 [03:49<5:56:45,  7.62s/it] || G1:29/2839 [03:53<6:08:42,  7.87s/it]
[t=5m09s] G0:35/2838 [04:26<5:53:53,  7.58s/it] || G1:33/2839 [04:23<5:49:35,  7.48s/it]
[t=5m41s] G0:38/2838 [04:50<6:02:03,  7.76s/it] || G1:37/2839 [04:52<5:37:53,  7.24s/it]
[t=6m13s] G0:42/28

C6 — clean per-group MD (direct join, no reconstruction) + validation


In [6]:
import pandas as pd, glob
kp = pd.read_csv("/kaggle/working/md_per_kp_full.csv")
kp["gen_rel"] = kp["gen"].str.extract(r"(Gen_results_FreeFine_2d/.+\.png)$")   # strip base_dir if absolute
meta = pd.read_csv(glob.glob("/kaggle/input/**/sample_metadata.csv", recursive=True)[0])
m = kp.merge(meta, on="gen_rel", how="left")
print("rows:", len(m), "| unmatched:", int(m["gen_rel"].isna().sum() + m["edit_type"].isna().sum()))
print("GLOBAL MD:", round(m["dist"].mean(), 4), " (must be ~8.4993)")
print("\n== MD by edit_type ==  (salvage: move 3.75 / resize 9.96 / rotate 10.43)")
print(m.groupby("edit_type")["dist"].mean().round(4))
print("\n== MD by difficulty ==  (salvage: easy 4.85 / medium 6.98 / hard 13.37)")
print(m.groupby("difficulty")["dist"].mean().round(4))
print("\n== MD by type x difficulty ==")
print(m.groupby(["edit_type","difficulty"])["dist"].mean().unstack().round(3))
m.groupby(["edit_type","difficulty"])["dist"].mean().round(4).to_csv("/kaggle/working/md_per_group_CLEAN.csv")
print("\nsaved md_per_group_CLEAN.csv")

rows: 120155 | unmatched: 0
GLOBAL MD: 8.5455  (must be ~8.4993)

== MD by edit_type ==  (salvage: move 3.75 / resize 9.96 / rotate 10.43)
edit_type
move       3.8366
resize     9.9939
rotate    10.4721
Name: dist, dtype: float64

== MD by difficulty ==  (salvage: easy 4.85 / medium 6.98 / hard 13.37)
difficulty
easy       4.8572
hard      13.5050
medium     6.9761
Name: dist, dtype: float64

== MD by type x difficulty ==
difficulty   easy    hard  medium
edit_type                        
move        3.286   4.750   3.547
resize      5.680  16.504   9.101
rotate      4.733  15.844   7.188

saved md_per_group_CLEAN.csv
